# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dwaynemongaya/flyrank-ml-internship_dwaynemongaya/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule

Recommend a page for refresh if it has meaningful search visibility (high Google Search Console impressions) and a strong average search position. These pages are already attracting users and may benefit from content improvements that preserve or increase their search performance.

### Reason code

- STALE_VISIBLE – The page is old and still receives search impressions.

### Action label

- REFRESH – Recommend the page for content review and refresh.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
from datasets import load_dataset
from google.colab import userdata
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    token=HF_TOKEN,
)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [7]:
import pandas as pd
from itertools import islice

# Load the first 5000 rows into a DataFrame
df = pd.DataFrame(list(islice(ds["train"], 5000)))

print(df.shape)
df.head()

(5000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 30 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   report_date               5000 non-null   object 
 1   client_hash_id            5000 non-null   object 
 2   content_hash_id           5000 non-null   object 
 3   client_has_gsc            5000 non-null   bool   
 4   client_has_ga4            5000 non-null   bool   
 5   gsc_data_available        5000 non-null   bool   
 6   ga4_data_available        5000 non-null   bool   
 7   gsc_impressions           5000 non-null   int64  
 8   gsc_clicks                5000 non-null   int64  
 9   gsc_sum_position          5000 non-null   int64  
 10  gsc_avg_position          5000 non-null   float64
 11  ga4_pageviews             5000 non-null   int64  
 12  ga4_sessions              5000 non-null   int64  
 13  ga4_users                 5000 non-null   int64  
 14  ga4_enga

In [10]:
import os

# Simple baseline score
df["baseline_score"] = (
    (df["gsc_impressions"] >= 100).astype(int) * 50 +
    (df["gsc_avg_position"] <= 10).astype(int) * 50
)

# Reason code
df["reason_code"] = "VISIBLE_PAGE"

# Action
df["action"] = "REFRESH"

# Rank pages
ranked = df.sort_values("baseline_score", ascending=False)

# Show Top 10
display(
    ranked[
        [
            "content_hash_id",
            "gsc_impressions",
            "gsc_avg_position",
            "baseline_score",
            "reason_code",
            "action",
        ]
    ].head(20)
)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved work/outputs/baseline_action_score.csv")

,content_hash_id,gsc_impressions,gsc_avg_position,baseline_score,reason_code,action
3825,content_213eb91f21a43550,424,1.099057,100,VISIBLE_PAGE,REFRESH
1565,content_84f5a9ecfefa108e,117,1.863248,100,VISIBLE_PAGE,REFRESH
2292,content_213eb91f21a43550,118,5.830508,100,VISIBLE_PAGE,REFRESH
3957,content_8bea9167e438fad6,108,5.777778,100,VISIBLE_PAGE,REFRESH
1150,content_37b3bafd5f88fdd1,119,5.000000,100,VISIBLE_PAGE,REFRESH
1221,content_f94fe855380e150f,305,1.586885,100,VISIBLE_PAGE,REFRESH
3420,content_aead070e1f002702,128,6.078125,100,VISIBLE_PAGE,REFRESH
4779,content_84f5a9ecfefa108e,104,1.692308,100,VISIBLE_PAGE,REFRESH
2352,content_f94fe855380e150f,171,2.555556,100,VISIBLE_PAGE,REFRESH
3766,content_438b962eaef05d1d,105,3.371429,100,VISIBLE_PAGE,REFRESH


Saved work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*



1. **Action:** REFRESH  
   - **Reason code:** VISIBLE_PAGE  
   - **Confidence:** High  
   - **Why:** The page has strong search visibility and a good average search position, making it a good candidate for a content refresh.  
   - **What would make it wrong:** The page may already be recently updated or fully optimized.

2. **Action:** REFRESH  
   - **Reason code:** VISIBLE_PAGE  
   - **Confidence:** High  
   - **Why:** The page continues to receive meaningful impressions and ranks well in search results.  
   - **What would make it wrong:** Traffic may be temporary due to seasonality or trending topics.

3. **Action:** REFRESH  
   - **Reason code:** VISIBLE_PAGE  
   - **Confidence:** High  
   - **Why:** The page satisfies the baseline rule by combining good search visibility with a strong ranking position.  
   - **What would make it wrong:** The page may already satisfy user intent and require no further updates.

4. **Action:** REFRESH  
   - **Reason code:** VISIBLE_PAGE  
   - **Confidence:** Medium  
   - **Why:** The page receives enough impressions to justify a review.  
   - **What would make it wrong:** The impressions may come from low-value search queries.

5. **Action:** REFRESH  
   - **Reason code:** VISIBLE_PAGE  
   - **Confidence:** Medium  
   - **Why:** The page performs well in search and could benefit from updated content.  
   - **What would make it wrong:** The page may already be performing at its maximum potential.

6. **Action:** REFRESH  
   - **Reason code:** VISIBLE_PAGE  
   - **Confidence:** Medium  
   - **Why:** The page remains visible in search results and attracts users.  
   - **What would make it wrong:** Recent improvements may not yet be reflected in the current data.

7. **Action:** REFRESH  
   - **Reason code:** VISIBLE_PAGE  
   - **Confidence:** Medium  
   - **Why:** The page has meaningful impressions and a strong average search position.  
   - **What would make it wrong:** Search rankings may change because of external algorithm updates rather than content quality.

8. **Action:** REFRESH  
   - **Reason code:** VISIBLE_PAGE  
   - **Confidence:** Medium  
   - **Why:** The page qualifies under the baseline scoring rule.  
   - **What would make it wrong:** User engagement may already be excellent, reducing the need for changes.

9. **Action:** REFRESH  
   - **Reason code:** VISIBLE_PAGE  
   - **Confidence:** Medium  
   - **Why:** The page consistently appears in search results and receives organic impressions.  
   - **What would make it wrong:** Search demand may naturally decline over time.

10. **Action:** REFRESH  
    - **Reason code:** VISIBLE_PAGE  
    - **Confidence:** Medium  
    - **Why:** The page remains visible to users and could benefit from updated content.  
    - **What would make it wrong:** The page may already meet all user expectations.

11. **Action:** REFRESH  
    - **Reason code:** VISIBLE_PAGE  
    - **Confidence:** Medium  
    - **Why:** The page satisfies the baseline rule and has continued search exposure.  
    - **What would make it wrong:** Improvements may not produce measurable ranking gains.

12. **Action:** REFRESH  
    - **Reason code:** VISIBLE_PAGE  
    - **Confidence:** Medium  
    - **Why:** Search visibility suggests the page is valuable enough to review.  
    - **What would make it wrong:** The page may already have high conversion performance despite unchanged content.

13. **Action:** REFRESH  
    - **Reason code:** VISIBLE_PAGE  
    - **Confidence:** Medium  
    - **Why:** The page has a competitive average position and receives search traffic.  
    - **What would make it wrong:** Ranking improvements may depend on factors outside the content itself.

14. **Action:** REFRESH  
    - **Reason code:** VISIBLE_PAGE  
    - **Confidence:** Medium  
    - **Why:** The page continues attracting organic search visitors.  
    - **What would make it wrong:** The content may already be considered authoritative.

15. **Action:** REFRESH  
    - **Reason code:** VISIBLE_PAGE  
    - **Confidence:** Medium  
    - **Why:** The page is visible enough that refreshing it could preserve search performance.  
    - **What would make it wrong:** Search visibility alone does not guarantee that content needs updating.

16. **Action:** REFRESH  
    - **Reason code:** VISIBLE_PAGE  
    - **Confidence:** Medium  
    - **Why:** The page matches the baseline scoring criteria.  
    - **What would make it wrong:** User satisfaction may already be very high.

17. **Action:** REFRESH  
    - **Reason code:** VISIBLE_PAGE  
    - **Confidence:** Medium  
    - **Why:** Strong impressions indicate that many users still discover this page.  
    - **What would make it wrong:** The traffic may not represent the page's long-term value.

18. **Action:** REFRESH  
    - **Reason code:** VISIBLE_PAGE  
    - **Confidence:** Medium  
    - **Why:** The page continues to rank well enough to justify review.  
    - **What would make it wrong:** Search intent may have changed since the page was created.

19. **Action:** REFRESH  
    - **Reason code:** VISIBLE_PAGE  
    - **Confidence:** Medium  
    - **Why:** The page remains competitive in search rankings.  
    - **What would make it wrong:** Content changes could unintentionally reduce current performance.

20. **Action:** REFRESH  
    - **Reason code:** VISIBLE_PAGE  
    - **Confidence:** Medium  
    - **Why:** The page is consistently visible and satisfies the baseline scoring rule.  
    - **What would make it wrong:** The baseline rule uses only a few observable signals and may overlook other important factors.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


Some recommendations may be weak because the baseline only considers search visibility and average position. A page may rank highly even if it was recently updated or already performs well enough that no refresh is needed.

I confirmed that the baseline does not use any label-derived fields, future performance information, or product-generated recommendation flags. The rule relies only on observable signals that are available at the decision time, reducing the risk of feature leakage.

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.